# Análise exploratória e métricas - Steam Player Data
Projeto final: Análise de Dados e BI (setor Games). Entrada: `steam_players_limpo.csv` (saída do notebook 01).

## 1. Carga dos dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]
df = pd.read_csv(nome_arquivo, parse_dates=["date"])
df = df.sort_values(["game_name", "date"]).reset_index(drop=True)

print(df.shape)
df.head()

## 2. Parâmetros
Limiares de risco definidos pelo grupo. Podem ser ajustados e a análise reexecutada.

In [ ]:
MESES_QUEDA_ALERTA = 3   # meses seguidos de queda para considerar a perda contínua
TAXA_QUEDA_ALERTA = 50   # % de perda em relação ao pico histórico da média de jogadores

## 3. Métricas do projeto
- **Variação percentual mensal:** `percent_gain` (já recalculada na limpeza).
- **Pico histórico:** maior média mensal de jogadores do jogo até aquele mês.
- **Taxa de queda (proxy de churn):** `(1 - média do mês / pico histórico) * 100`.
- **Meses consecutivos de queda:** quantos meses seguidos a média caiu em relação ao mês anterior.

In [ ]:
df["em_queda"] = df["gain"] < 0

def contar_sequencia(serie):
    # Zera a contagem a cada mês sem queda
    grupo = (~serie).cumsum()
    return serie.groupby(grupo).cumsum()

df["meses_consecutivos_queda"] = (
    df.groupby("game_name")["em_queda"].transform(contar_sequencia).astype(int)
)

df["pico_historico"] = df.groupby("game_name")["avg_players"].cummax()
df["taxa_queda_pico"] = (1 - df["avg_players"] / df["pico_historico"]) * 100

df[["game_name", "date", "avg_players", "percent_gain", "pico_historico",
    "taxa_queda_pico", "meses_consecutivos_queda"]].tail()

## 4. Classificação de risco
- **Alto:** perda contínua (mínimo de meses seguidos de queda) e queda grande em relação ao pico.
- **Atenção:** apenas uma das duas condições.
- **Estável:** nenhuma das duas.

É um indicador de risco, não uma previsão de encerramento: a base só contém jogos do ranking atual, sem jogos já encerrados para validar os limiares.

In [ ]:
cond_sequencia = df["meses_consecutivos_queda"] >= MESES_QUEDA_ALERTA
cond_queda = df["taxa_queda_pico"] >= TAXA_QUEDA_ALERTA

df["risco"] = np.select(
    [cond_sequencia & cond_queda, cond_sequencia | cond_queda],
    ["Alto", "Atenção"],
    default="Estável",
)

ultimo_mes = df["date"].max()
atual = df[df["date"] == ultimo_mes].copy()
print("Último mês da base:", ultimo_mes.date())
print(atual["risco"].value_counts())

## 5. Estatística descritiva

In [ ]:
atual[["avg_players", "taxa_queda_pico", "meses_consecutivos_queda"]].describe().round(2)

In [ ]:
# Jogos com maior queda em relação ao pico no último mês
top_queda = atual.sort_values("taxa_queda_pico", ascending=False).head(10)

plt.figure(figsize=(9, 5))
sns.barplot(data=top_queda, y="game_name", x="taxa_queda_pico", color="steelblue")
plt.xlabel("Queda em relação ao pico histórico (%)")
plt.ylabel("")
plt.title("Jogos com maior queda em relação ao pico (último mês)")
plt.tight_layout()
plt.show()

## 6. Sazonalidade
Mediana da variação percentual por mês do calendário, considerando todos os jogos. Mediana em vez de média para reduzir o efeito de picos de lançamento.

In [ ]:
df["mes_calendario"] = df["date"].dt.month
sazonal = df.dropna(subset=["percent_gain"]).groupby("mes_calendario")["percent_gain"].median()

nomes_mes = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
plt.figure(figsize=(9, 4))
plt.bar(nomes_mes, sazonal.values, color="steelblue")
plt.axhline(0, color="gray", linewidth=0.8)
plt.ylabel("Mediana da variação mensal (%)")
plt.title("Sazonalidade da base ativa (todos os jogos)")
plt.tight_layout()
plt.show()

print(sazonal.round(2))

## 7. Correlação
Features por jogo no último mês. Correlação indica associação, não causa.

In [ ]:
perfil = atual.set_index("game_name")[["avg_players", "taxa_queda_pico", "meses_consecutivos_queda"]].copy()
perfil["idade_meses"] = df.groupby("game_name")["date"].count()
perfil["volatilidade"] = df.groupby("game_name")["percent_gain"].std()
perfil["maior_sequencia_queda"] = df.groupby("game_name")["meses_consecutivos_queda"].max()
perfil["log_avg_players"] = np.log10(perfil["avg_players"])

corr = perfil[["idade_meses", "log_avg_players", "volatilidade",
               "taxa_queda_pico", "maior_sequencia_queda"]].corr().round(2)

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlação entre características dos jogos")
plt.tight_layout()
plt.show()

## 8. Evolução em relação ao pico
Jogos classificados como risco Alto no último mês, mostrando a média de jogadores como % do pico histórico.

In [ ]:
jogos_alto = atual[atual["risco"] == "Alto"]["game_name"].tolist()
print("Jogos em risco Alto:", jogos_alto)

plt.figure(figsize=(10, 5))
for jogo in jogos_alto[:6]:
    d = df[df["game_name"] == jogo]
    plt.plot(d["date"], 100 - d["taxa_queda_pico"], label=jogo)
plt.ylabel("Média de jogadores (% do pico histórico)")
plt.legend(fontsize=8)
plt.title("Evolução dos jogos em risco Alto")
plt.tight_layout()
plt.show()

## 9. Exportação para o dashboard
- `steam_metricas_mensal.csv`: uma linha por jogo e mês, com as métricas e o risco.
- `steam_resumo_jogos.csv`: uma linha por jogo, com a situação no último mês.

In [ ]:
mensal = df[["game_name", "date", "avg_players", "peak_players", "percent_gain",
             "pico_historico", "taxa_queda_pico", "meses_consecutivos_queda", "risco"]].copy()
mensal["date"] = mensal["date"].dt.strftime("%Y-%m-%d")
mensal = mensal.round({"avg_players": 0, "percent_gain": 2, "pico_historico": 0, "taxa_queda_pico": 2})
mensal.to_csv("steam_metricas_mensal.csv", index=False)

resumo = atual[["game_name", "avg_players", "pico_historico", "taxa_queda_pico",
                "meses_consecutivos_queda", "risco"]].copy()
resumo["maior_sequencia_queda"] = resumo["game_name"].map(perfil["maior_sequencia_queda"])
resumo["idade_meses"] = resumo["game_name"].map(perfil["idade_meses"])
resumo = resumo.round({"avg_players": 0, "pico_historico": 0, "taxa_queda_pico": 2})
resumo.to_csv("steam_resumo_jogos.csv", index=False)

files.download("steam_metricas_mensal.csv")
files.download("steam_resumo_jogos.csv")